# Projet Nexovia

Maroua Nabih, Mathis Raffin et Axel Chasteauneuf — Mines Nancy, 2026–2027.

Ce notebook présente les méthodes et quelques exemples de planification. Les fonctions sont importées depuis `code/`, où se trouve leur implémentation.

Exécuter les cellules dans l’ordre, depuis la racine du dépôt. Les campagnes longues sont désactivées par défaut avec `LANCER_CAMPAGNES = False`. Voir le [guide d’utilisation](docs/UTILISATION.md) pour les expériences complètes.

Une assistance IA a été utilisée au cours du projet, comme indiqué dans les documents de rendu.


In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, "code")
LANCER_CAMPAGNES = False


## 1. Socle commun

Ces fonctions sont utilisées par toutes les méthodes : elles lisent une instance, vérifient qu'un planning respecte toutes les contraintes et calculent son coût f = D + c_rep · R (distance totale plus coût d'un report fois le nombre de reports).

### 1.1 Lecture des données

Chaque instance est un dossier de six fichiers CSV (techniciens, interventions, disponibilités, lieux, trajets, paramètres). `lire_instance` les charge dans des tableaux pandas. On lit ensuite la petite instance N0 et on affiche ses tableaux pour vérifier la lecture.

In [ ]:
import pandas as pd
from pathlib import Path
from socle import lire_instance


In [ ]:
# lecture de la petite instance N0 (8 interventions, 3 techniciens)
N0 = lire_instance("donnees/N0")

In [ ]:
# on affiche les tableaux pour vérifier que la lecture s'est bien passée
print("Techniciens :")
display(N0["techniciens"])

print("Interventions :")
display(N0["interventions"])

print("Disponibilités :")
display(N0["disponibilites"])

print("Paramètres :")
display(N0["parametres"])

In [ ]:
display(N0["trajets"].head())

### 1.2 Préparation des données

Filtrer un tableau pandas est lent, et les méthodes vérifient des millions de tournées. `preparer_instance` recopie donc une fois pour toutes les données dans des dictionnaires : distance et temps pour chaque couple de lieux, fenêtre horaire, durée et compétence de chaque intervention, domicile, compétences et créneaux de chaque technicien, et coût d'un report.

In [ ]:
from socle import preparer_instance


In [ ]:
# petit test : on vérifie quelques valeurs après la préparation
preparer_instance(N0)

print("Distance H1 → I1 :", N0["dist"][("H1", "I1")], "km")
print("Temps H1 → I1 :", N0["temps"][("H1", "I1")], "min")
print("Disponibilités A2 :", N0["dispos"]["A2"])
print("Pénalité :", N0["penalite"])

### 1.3 Vérification d'un planning et fonction objectif

- `verifier_missions` : chaque intervention apparaît exactement une fois (dans une tournée ou dans les reportées) ;
- `verifier_competences` : chaque technicien a la compétence de ses interventions ;
- `faire_trajet` : calcule l'heure d'arrivée d'un trajet, qui ne peut se faire que pendant les créneaux du technicien (un trajet peut être coupé par une pause) ;
- `verifier_tournee` : vérifie une seule tournée (compétences, créneaux, fenêtres horaires, retour au domicile). C'est la fonction la plus utilisée : les méthodes ne revérifient que les tournées qu'elles viennent de modifier ;
- `distance_tournee` et `f` : distance d'une tournée et coût total d'un planning ;
- `est_realisable` : vérification complète, utilisée pour contrôler les solutions finales.

In [ ]:
from socle import verifier_missions


In [ ]:
from socle import verifier_competences


In [ ]:
from socle import lire_disponibilites


In [ ]:
from socle import faire_trajet, verifier_tournee, distance_tournee, f
from socle import EPS


In [ ]:
from socle import est_realisable


### 1.4 Test sur une solution construite à la main

On vérifie les fonctions sur une solution de N0 faite à la main : elle doit être réalisable, et on calcule la distance de chaque tournée puis son coût.

In [ ]:
# solution construite à la main au premier cours, pour tester les fonctions
S0 = {
    "A1": ["I1", "I4", "I2"],
    "A2": ["I3", "I6"],
    "A3": ["I5", "I8"],
    "reportees": ["I7"]
}

print("Solution réalisable :", est_realisable(S0, N0))

In [ ]:
# distance de chaque tournée de cette solution
print("Distance A1 :", distance_tournee(N0, "A1", S0["A1"]))
print("Distance A2 :", distance_tournee(N0, "A2", S0["A2"]))
print("Distance A3 :", distance_tournee(N0, "A3", S0["A3"]))

In [ ]:
# coût total = distance + pénalité x nombre de reports
print("valeur de s0 pour N0 :", round(f(S0, N0), 2))

## 2. Heuristique gloutonne

L'heuristique construit la solution de départ S₀. Les interventions sont prises par fin de fenêtre croissante (les plus urgentes d'abord). Chacune est insérée chez le technicien et à la position qui ajoutent le moins de kilomètres, en ne vérifiant que la tournée modifiée. Si aucune insertion n'est possible, l'intervention est reportée.

In [ ]:
from socle import heuristique_gloutonne


In [ ]:
# solution de départ obtenue par le glouton sur N0
S0 = heuristique_gloutonne(N0)

print("Solution initiale :")

for tech in N0["techniciens"]["technicien"]:
    print(tech, ":", S0[tech])

print("Interventions reportées :", S0["reportees"])

print("Solution réalisable :", est_realisable(S0, N0))

print("Coût initial :", round(f(S0, N0), 2))

In [ ]:
# glouton sur N10
N10 = lire_instance("donnees/N10")
preparer_instance(N10)

S10 = heuristique_gloutonne(N10)

print("Solution N10 :", S10)
print("Réalisable :", est_realisable(S10, N10))
print("Coût initial :", round(f(S10, N10), 2))

In [ ]:
# le coût d'un report est la 2e valeur de parametres.csv
print("Pénalité N0 :", N0["parametres"]["valeur"].iloc[1])
print("Pénalité N10 :", N10["parametres"]["valeur"].iloc[1])

### 2.1 Toutes les instances

On lit et on prépare les neuf instances, puis on applique le glouton à chacune : coût, nombre de reports, faisabilité et temps de calcul sont rassemblés dans un tableau.

In [ ]:
noms = ["N0", "N10", "N20", "N30", "N50", "N80", "N100", "N150", "N200"]

# vérification : nombre d'interventions de chaque instance
for nom in noms:
    N = lire_instance("donnees/" + nom)
    print(nom, ":", len(N["interventions"]), "interventions")

In [ ]:
# on lit toutes les instances et on les garde dans un dictionnaire
noms = ["N20", "N30", "N50", "N80", "N100", "N150", "N200"]

instances = {}

for nom in noms:
    instances[nom] = lire_instance("donnees/" + nom)

In [ ]:
# préparation des dictionnaires (distances, fenêtres, compétences...)
for nom in noms:
    N = instances[nom]
    preparer_instance(N)

print("Toutes les instances sont préparées.")

In [ ]:
# contrôle que la préparation a bien été faite
for nom in noms:
    print(nom, ":", "techs" in instances[nom])

In [ ]:
# Glouton sur toutes les instances : coût, nombre de reports, temps de calcul
import time

solutions_gloutonnes = {}
resultats_glouton = []

for nom in noms:

    N = instances[nom]

    debut = time.perf_counter()
    S = heuristique_gloutonne(N)
    duree = time.perf_counter() - debut

    solutions_gloutonnes[nom] = S

    cout = round(f(S, N), 2)
    reports = len(S["reportees"])
    realisable = est_realisable(S, N)

    resultats_glouton.append({
        "Instance": nom,
        "Coût initial": cout,
        "Reports": reports,
        "Réalisable": realisable,
        "Temps (s)": round(duree, 2)
   })

    print(nom, ":", cout, "| Reports :", reports,
          "| Réalisable :", realisable, "| Temps :", round(duree, 2), "s")

In [ ]:
# tableau récapitulatif du glouton, enregistré en csv
import pandas as pd

tableau_glouton = pd.DataFrame(resultats_glouton)

display(tableau_glouton)

tableau_glouton.to_csv(
    "resultats_glouton.csv",
    index=False,
    sep=";"
)

In [ ]:
# (on refait la préparation par sécurité)
for nom in instances:
    preparer_instance(instances[nom])

print("Toutes les instances sont préparées.")

In [ ]:
for nom in noms:
    print(nom, ":", "techs" in instances[nom])

In [ ]:
# on ajoute N0 et N10, préparées plus haut, au dictionnaire des instances
instances["N0"] = N0
instances["N10"] = N10

solutions_gloutonnes["N0"] = S0
solutions_gloutonnes["N10"] = S10
# Départs identiques pour les comparaisons finales ; distincts des gloutons exploratoires.
import json
with open("donnees/S0_communs.json", encoding="utf-8") as entree:
    solutions_communes = json.load(entree)


### 2.2 Affichage d'un planning

`afficher_planning` refait la journée de chaque technicien et affiche les heures de départ, de chaque trajet, de chaque intervention et du retour au domicile. Cela permet de vérifier une solution à la main.

In [ ]:
# convertit des minutes (ex : 495.5) en texte hh:mm:ss pour l'affichage
def afficher_heure(minutes):
    secondes = round(minutes * 60)
    h = secondes // 3600
    m = (secondes % 3600) // 60
    s = secondes % 60

    return f"{h:02d}:{m:02d}:{s:02d}"

In [ ]:
# Affiche le planning d'une solution réalisable : heure de départ, trajets,
# heure de chaque mission et retour au domicile, pour chaque technicien.
def afficher_planning(S, N):

    if not est_realisable(S, N):
        print("La solution n'est pas réalisable")
        return

    for tech in N["techniciens"]["technicien"]:

        print("\nTechnicien :", tech)

        ligne = N["techniciens"][
            N["techniciens"]["technicien"] == tech
        ].iloc[0]

        domicile = ligne["domicile"]
        disponibilites = lire_disponibilites(N, tech)

        heure = disponibilites[0][0]
        lieu = domicile

        if len(S[tech]) == 0:
            print("Aucune mission")
            continue

        print("Départ :", afficher_heure(heure))

        for mission in S[tech]:

            # Déplacement vers la mission
            depart = heure
            heure = faire_trajet(N, lieu, mission, heure, disponibilites)

            print("Trajet", lieu, "→", mission, ":",
                  afficher_heure(depart), "à", afficher_heure(heure))

            intervention = N["interventions"][
                N["interventions"]["intervention"] == mission
            ].iloc[0]

            # Heure de début et de fin de la mission
            for debut, fin in disponibilites:

                debut_mission = max(
                    heure, debut, intervention["debut_fenetre_min"]
                )

                fin_mission = debut_mission + intervention["duree_min"]

                if fin_mission <= fin + EPS and fin_mission <= intervention["fin_fenetre_min"] + EPS:
                    print("Mission", mission, ":",
                          afficher_heure(debut_mission), "à",
                          afficher_heure(fin_mission))
                    heure = fin_mission
                    break

            lieu = mission

        # Retour à la maison
        heure = faire_trajet(N, lieu, domicile, heure, disponibilites)

        print("Retour au domicile :", afficher_heure(heure))

    print("\nInterventions reportées :", S["reportees"])

In [ ]:
# planning de la solution gloutonne de N0
afficher_planning(S0, N0)

In [ ]:
afficher_planning(S10, N10)

## 3. Recherches locales

### 3.1 Voisinage relocate

Un voisin est obtenu en retirant une intervention de sa liste (une tournée ou les reportées) et en la remettant ailleurs : à une autre position, chez un autre technicien compétent, ou dans les reportées. `generer_voisins` donne les voisins réalisables un par un, sans les garder tous en mémoire, et ne revérifie que les deux tournées modifiées.

In [ ]:
from local_tabou import generer_voisins


In [ ]:
from local_tabou import voisinage


In [ ]:
# test du voisinage sur N0 : nombre de voisins et les 3 premiers
voisins, mouvements = voisinage(S0, N0)

print("Nombre de voisins réalisables :", len(voisins))

for i in range(min(3, len(voisins))):
    print("\nVoisin", i + 1)
    print("Mouvement :", mouvements[i])
    print("Solution :", voisins[i])
    print("Coût :", round(f(voisins[i], N0), 2))

In [ ]:
# contrôle : tous les voisins renvoyés doivent être réalisables
for S, N in [(S0, N0), (S10, N10)]:

    voisins, mouvements = voisinage(S, N)

    print("Nombre de voisins :", len(voisins))
    print("Tous réalisables :", all(est_realisable(V, N) for V in voisins))

### 3.2 Descentes

- **Première amélioration** : on parcourt les voisins et on se déplace vers le premier qui fait baisser le coût ;
- **Plus forte pente** : on examine tous les voisins et on se déplace vers le meilleur.

Les deux s'arrêtent dans un optimum local, quand aucun voisin n'est meilleur. On les compare sur N0 et N10.

In [ ]:
from local_tabou import premiere_amelioration


In [ ]:
from local_tabou import plus_forte_pente


In [ ]:
# comparaison des deux descentes sur N0
S_premiere = premiere_amelioration(S0, N0)
S_pente = plus_forte_pente(S0, N0)

print("Coût initial :", round(f(S0, N0), 2))
print("Première amélioration :", round(f(S_premiere, N0), 2))
print("Plus forte pente :", round(f(S_pente, N0), 2))

print("Réalisable première amélioration :", est_realisable(S_premiere, N0))
print("Réalisable plus forte pente :", est_realisable(S_pente, N0))

In [ ]:
# comparaison des deux descentes sur N10
S_premiere10 = premiere_amelioration(S10, N10)
S_pente10 = plus_forte_pente(S10, N10)

print("Coût initial :", round(f(S10, N10), 2))
print("Première amélioration :", round(f(S_premiere10, N10), 2))
print("Plus forte pente :", round(f(S_pente10, N10), 2))

print("Réalisable première amélioration :", est_realisable(S_premiere10, N10))
print("Réalisable plus forte pente :", est_realisable(S_pente10, N10))

In [ ]:
# on regarde si les deux descentes arrivent à la même solution
print("Première amélioration :")
for cle, missions in S_premiere10.items():
    print(cle, ":", missions)

print("\nPlus forte pente :")
for cle, missions in S_pente10.items():
    print(cle, ":", missions)

print("\nSolutions identiques :", S_premiere10 == S_pente10)

### 3.3 Campagne finale sur les mêmes départs

Les deux descentes sont évaluées depuis `donnees/S0_communs.json`. Les gloutons recalculés plus haut sont conservés comme exemples et départs exploratoires. Cette campagne est facultative et désactivée par défaut.


In [ ]:
if LANCER_CAMPAGNES:
    # première amélioration sur toutes les instances, depuis la solution gloutonne
    import time

    noms = ["N0", "N10", "N20", "N30", "N50", "N80", "N100", "N150", "N200"]

    solutions_premiere = {}

    for nom in noms:

        N = instances[nom]
        S = solutions_communes[nom]

        # Première amélioration
        debut = time.perf_counter()
        S_finale = premiere_amelioration(S, N)
        temps = time.perf_counter() - debut

        solutions_premiere[nom] = S_finale

        print(nom)
        print("Coût final :", round(f(S_finale, N), 2))
        print("Reports :", len(S_finale["reportees"]))
        print("Temps :", round(temps, 2), "secondes")
        print("Réalisable :", est_realisable(S_finale, N))
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # plus forte pente sur toutes les instances
    import time

    solutions_pente = {}
    resultats_pente = []

    for nom in noms:

        N = instances[nom]
        S = solutions_communes[nom]

        # On applique la plus forte pente
        debut = time.perf_counter()
        S_finale = plus_forte_pente(S, N)
        temps = time.perf_counter() - debut

        solutions_pente[nom] = S_finale

        cout = round(f(S_finale, N), 2)
        reports = len(S_finale["reportees"])
        realisable = est_realisable(S_finale, N)

        resultats_pente.append({
            "Instance": nom,
            "Coût final": cout,
            "Reports": reports,
            "Réalisable": realisable,
            "Temps (s)": round(temps, 2)
        })

        print(nom, ":", cout, "| Reports :", reports,
              "| Réalisable :", realisable,
              "| Temps :", round(temps, 2), "s", flush=True)
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # Gains de la comparaison finale : même référence pour toutes les méthodes.
    for nom in noms:
        N = instances[nom]
        cout_initial = f(solutions_communes[nom], N)
        cout_premiere = f(solutions_premiere[nom], N)
        cout_pente = f(solutions_pente[nom], N)
        print(nom, "| S0 commun :", round(cout_initial, 2),
              "| Première :", round(100 * (cout_initial - cout_premiere) / cout_initial, 2), "%",
              "| Pente :", round(100 * (cout_initial - cout_pente) / cout_initial, 2), "%")

else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


## 4. Métaheuristiques

### 4.1 Recuit simulé

Le fichier `recuit.py` contient trois fonctions :

- `voisin_aleatoire` : au lieu de construire tout le voisinage, on tire au hasard les quatre éléments d'un mouvement relocate (liste de départ, intervention, destination, position) et on ne vérifie que les deux tournées modifiées ;
- `calculer_T0` : la température initiale est choisie pour qu'une dégradation moyenne autour de S₀ soit acceptée avec la probabilité τ₀, soit T₀ = -Δ⁺ / ln τ₀ ;
- `recuit_simule` : à chaque itération, un voisin est accepté s'il améliore le coût, sinon avec la probabilité exp(−Δ/T). La température est multipliée par α tous les k · n itérations ; quand elle passe sous T₀/1000, on repart de la meilleure solution avec T = T₀ (réchauffage). L'arrêt se fait au bout du temps imparti. Le coût d'un voisin est calculé en ne recalculant que la distance des tournées modifiées.

Réglage retenu : τ₀ = 0,1, α = 0,99, k = 20, 180 secondes.

Implémentation : [code/recuit.py](code/recuit.py).

Exemple : le recuit sur N50.

In [ ]:
import json
import recuit

N50 = lire_instance("donnees/N50")
preparer_instance(N50)
S0_N50 = json.load(open("donnees/S0_communs.json"))["N50"]

S, cout, infos = recuit.recuit_simule(S0_N50, N50, temps_max=5, graine=1)
print("Coût de départ :", round(f(S0_N50, N50), 2), "| coût final :", round(cout, 2))
print("Itérations :", infos["iterations"], "| T0 :", round(infos["T0"], 2), "| réalisable :", est_realisable(S, N50))

### 4.2 Recherche tabou

À chaque itération, on parcourt le voisinage relocate et on garde 40 voisins non tabous tirés au hasard (tirage par réservoir). On se déplace vers le meilleur des 40, même s'il est moins bon que la solution actuelle. Le mouvement inverse (remettre l'intervention chez son ancien technicien) devient alors tabou pendant L itérations. Chaque appel réalise cinq essais de 180 secondes et relève le meilleur coût à 60, 120 et 180 secondes.

In [ ]:
from local_tabou import recherche_tabou


In [ ]:
if LANCER_CAMPAGNES:
    # premier test du tabou sur N0 avec L = 10
    N = instances["N0"]
    S0 = solutions_communes["N0"]

    resultats_N0, solutions_N0 = recherche_tabou(S0, N, L=10)

    display(pd.DataFrame(resultats_N0))

    print("Coût initial :", round(f(S0, N), 2))
    print("Meilleur coût :", min(r["Coût 180"] for r in resultats_N0))
    print("Toutes réalisables :",
          all(est_realisable(S, N) for S in solutions_N0))
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # même test sur N10
    N = instances["N10"]
    S0 = solutions_communes["N10"]

    resultats_N10, solutions_N10 = recherche_tabou(S0, N, L=10)

    display(pd.DataFrame(resultats_N10))

    print("Coût initial :", round(f(S0, N), 2))
    print("Meilleur coût :", min(r["Coût 180"] for r in resultats_N10))
    print("Toutes réalisables :",
          all(est_realisable(S, N) for S in solutions_N10))
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


**Réglage de la longueur L de la liste tabou.** Les valeurs 10, 20 et 30 sont testées sur six instances d'apprentissage ; la valeur qui donne le meilleur gain moyen à 180 secondes est ensuite évaluée sur les trois autres instances.

In [ ]:
if LANCER_CAMPAGNES:
    # Sensibilité de la longueur L de la liste tabou (10, 20, 30) sur les instances
    # d'apprentissage. On sauvegarde après chaque instance pour ne rien perdre si le calcul est interrompu.
    train = ["N0", "N10", "N30", "N80", "N100", "N200"]

    sensibilite = []
    solutions_train = {}

    for L in [10, 20, 30]:

        for nom in train:

            N = instances[nom]
            S0 = solutions_communes[nom]
            cout_initial = f(S0, N)

            # Réutiliser les résultats déjà calculés
            if L == 10 and nom == "N0":
                resultats, solutions = resultats_N0, solutions_N0

            elif L == 10 and nom == "N10":
                resultats, solutions = resultats_N10, solutions_N10

            else:
                resultats, solutions = recherche_tabou(S0, N, L)

            solutions_train[(nom, L)] = solutions

            for r in resultats:

                sensibilite.append({
                    "Instance": nom,
                    "L": L,
                    "Essai": r["Essai"],
                    "Coût 60": r["Coût 60"],
                    "Coût 120": r["Coût 120"],
                    "Coût 180": r["Coût 180"],
                    "Gain 60": 100 * (cout_initial - r["Coût 60"]) / cout_initial,
                    "Gain 120": 100 * (cout_initial - r["Coût 120"]) / cout_initial,
                    "Gain 180": 100 * (cout_initial - r["Coût 180"]) / cout_initial,
                    "Reports": r["Reports"],
                    "Temps": r["Temps"]
                })

            print(nom, "| L =", L, "terminé", flush=True)

            # Sauvegarde après chaque instance
            pd.DataFrame(sensibilite).to_csv(
                "sensibilite_tabou.csv", index=False
            )
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # gain moyen par valeur de L : on retient celle qui donne le meilleur gain à 180 s
    tableau = pd.DataFrame(sensibilite)

    comparaison = tableau.groupby("L")[
        ["Gain 60", "Gain 120", "Gain 180"]
    ].mean().round(2)

    display(comparaison)

    L_choisi = int(comparaison["Gain 180"].idxmax())

    print("Valeur de L retenue :", L_choisi)
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # test de la valeur retenue sur les instances qui n'ont pas servi au réglage
    test = ["N20", "N50", "N150"]

    resultats_test = []
    solutions_test = {}

    for nom in test:

        N = instances[nom]
        S0 = solutions_communes[nom]

        resultats, solutions = recherche_tabou(S0, N, L_choisi)

        solutions_test[nom] = solutions

        for r in resultats:

            resultats_test.append({
                "Instance": nom,
                "Essai": r["Essai"],
                "Coût 60": r["Coût 60"],
                "Coût 120": r["Coût 120"],
                "Coût 180": r["Coût 180"],
                "Reports": r["Reports"],
                "Temps": r["Temps"]
            })

        print(nom, "| Réalisable :",
              all(est_realisable(S, N) for S in solutions),
              flush=True)

        pd.DataFrame(resultats_test).to_csv(
            "test_tabou.csv", index=False
        )

    display(pd.DataFrame(resultats_test))
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # résumé final du tabou (instances d'apprentissage + de test)
    resultats_train = tableau[tableau["L"] == L_choisi]

    resultats_finaux = pd.concat([
        resultats_train,
        pd.DataFrame(resultats_test)
    ])

    resume = resultats_finaux.groupby("Instance").agg(
        Meilleur_cout=("Coût 180", "min"),
        Cout_moyen=("Coût 180", "mean"),
        Ecart_type=("Coût 180", "std"),
        Temps_moyen=("Temps", "mean")
    ).round(2)

    display(resume)

    resume.to_csv("resume_tabou.csv")
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


**Contrôle du blocage.** Ce test montre si tous les mouvements deviennent tabous. L’arrêt anticipé sur les petites instances est une limite connue de cette version, sans aspiration ; ce contrôle ne prétend pas supprimer le blocage.


In [ ]:
# Vérifie que la liste tabou ne bloque pas la recherche : sur 50 itérations,
# on compte les voisins et ceux qui restent autorisés. Si aucun n'est autorisé, c'est bloqué.
import random
from collections import deque

def verifier_blocage(nom, L=30):

    N = instances[nom]
    S = {cle: solutions_gloutonnes[nom][cle][:] for cle in solutions_gloutonnes[nom]}

    liste_tabou = deque(maxlen=L)
    rng = random.Random(1)

    for iteration in range(1, 51):

        candidats = []
        total = 0
        autorises = 0

        for V, mouvement in generer_voisins(S, N):

            total += 1

            cle = (mouvement["mission"], mouvement["destination"])

            if cle in liste_tabou:
                continue

            autorises += 1

            if len(candidats) < 40:
                candidats.append((V, mouvement))
            else:
                j = rng.randrange(autorises)
                if j < 40:
                    candidats[j] = (V, mouvement)

        print("Itération :", iteration,
              "| Voisins :", total,
              "| Autorisés :", autorises)

        if not candidats:
            print("ARRÊT : aucun voisin autorisé.")
            return

        rng.shuffle(candidats)
        S, mouvement = min(candidats, key=lambda x: f(x[0], N))

        liste_tabou.append((
            mouvement["mission"],
            mouvement["source"]
        ))

    print("50 itérations terminées sans blocage.")

In [ ]:
# test du blocage sur les deux petites instances, avec la plus longue liste (L = 30)
print("=== N0 ===")
verifier_blocage("N0", L=30)

print("\n=== N10 ===")
verifier_blocage("N10", L=30)

### 4.3 Variante hybride ALNS + tabou

**Amelioration par IA**

À chaque itération, une partie de la solution est détruite (10 à 25 % des interventions sont reportées, au hasard ou parmi les plus coûteuses), puis réparée (insertion gloutonne ou par regret), avant quelques pas de recherche tabou. Les quatre couples destruction / réparation ont un poids qui augmente lorsqu'ils donnent de bons résultats, et une solution moins bonne peut être acceptée pour explorer une autre région.

In [ ]:
from local_tabou import copier_solution, detruire_hybride, reparer_hybride, hybride_alns_tabou


In [ ]:
# essai rapide de l'hybride sur N10 (10 s)
N = instances["N10"]
S0 = solutions_communes["N10"]

S_hybride, infos = hybride_alns_tabou(
    S0, N,
    L=30,
    temps_max=10,
    graine=1
)

print("Coût glouton :", round(f(S0, N), 2))
print("Coût hybride :", round(f(S_hybride, N), 2))
print("Nombre de reports :", len(S_hybride["reportees"]))
print("Réalisable :", est_realisable(S_hybride, N))
print("Temps :", infos["temps"], "secondes")
print("Itérations :", infos["iterations"])

In [ ]:
if LANCER_CAMPAGNES:
    # 5 essais de 10 s sur N10
    import pandas as pd

    N = instances["N10"]
    S0 = solutions_communes["N10"]

    resultats_hybride = []

    for essai in range(1, 6):

        S, infos = hybride_alns_tabou(
            S0, N,
            L=30,
            temps_max=10,
            graine=essai
        )

        resultats_hybride.append({
            "Essai": essai,
            "Coût": round(f(S, N), 2),
            "Reports": len(S["reportees"]),
            "Temps": infos["temps"],
            "Réalisable": est_realisable(S, N)
        })

    display(pd.DataFrame(resultats_hybride))
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # essais rapides de 10 s sur les instances de test
    import pandas as pd

    resultats_hybride_test = []

    for nom in ["N20", "N50", "N150"]:

        N = instances[nom]
        S0 = solutions_communes[nom]

        for essai in range(1, 6):

            S, infos = hybride_alns_tabou(
                S0, N,
                L=30,
                temps_max=10,
                graine=essai
            )

            resultats_hybride_test.append({
                "Instance": nom,
                "Essai": essai,
                "Coût": round(f(S, N), 2),
                "Reports": len(S["reportees"]),
                "Temps": infos["temps"],
                "Réalisable": est_realisable(S, N)
            })

        print(nom, "terminé", flush=True)

        pd.DataFrame(resultats_hybride_test).to_csv(
            "hybride_test_rapide.csv", index=False
        )

    display(pd.DataFrame(resultats_hybride_test))
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


In [ ]:
if LANCER_CAMPAGNES:
    # un essai complet de 180 s sur les instances de test
    for nom in ["N20", "N50", "N150"]:

        N = instances[nom]
        S0 = solutions_communes[nom]

        S, infos = hybride_alns_tabou(
            S0, N,
            L=30,
            temps_max=180,
            graine=1
        )

        print(nom)
        print("Coût hybride :", round(f(S, N), 2))
        print("Reports :", len(S["reportees"]))
        print("Réalisable :", est_realisable(S, N))
        print("Temps :", infos["temps"], "s")
        print("--------------------", flush=True)
else:
    print("Campagne longue désactivée ; activer LANCER_CAMPAGNES pour la lancer.")


### 4.4 Algorithme génétique

Un individu est un planning complet. Le fichier `genetique.py` contient :

- `individu_aleatoire_ag` : un nouvel individu, construit comme le glouton mais avec un ordre des interventions mélangé et des insertions choisies parmi les trois meilleures ;
- `croisement_ag` : l'enfant hérite, pour chaque technicien, de la tournée de l'un des deux parents, sans copier deux fois une intervention ; les interventions restantes sont ensuite réinsérées (réparation) ;
- `mutation_ag` : déplacement ou échange d'interventions, en gardant un planning réalisable ;
- `algorithme_genetique` : population de 20 individus, sélection par tournoi de 2, croisement avec la probabilité 0,8, mutation avec la probabilité 0,25, conservation des deux meilleurs et refus des doublons. Une partie de chaque génération est faite d'individus nouveaux (20 %, puis 40 % avec une mutation plus forte après dix générations sans amélioration). La diversité de la population est suivie au cours du temps.

Implémentation : [code/genetique.py](code/genetique.py).

Exemple : le génétique sur N50.

In [ ]:
import genetique

resultats, solutions = genetique.algorithme_genetique(S0_N50, N50, budgets=(5,), graines=(1,))
print("Coût final :", round(resultats[0]["Coût 5"], 2), "| générations :", resultats[0]["Générations"])

## 5. Approche multiobjectif

On minimise maintenant séparément la distance D et le nombre de reports R.

### 5.1 Méthode ε-contrainte

On résout une suite de problèmes « minimiser D sous R ≤ ε » avec le recuit simulé, pour 11 valeurs de ε entre 0 et n (toutes les valeurs si n ≤ 12) :

- la contrainte est remplacée par une pénalité D + M · max(0, R − ε), avec M = 100 c_rep ;
- les valeurs de ε sont traitées dans l'ordre croissant, chaque recuit partant de la solution du ε précédent ;
- les 180 secondes sont partagées entre les ε, et la température décroît selon le temps disponible pour faire un cycle complet ;
- on garde les points non dominés et on mesure la qualité du front par son hypervolume.

Implémentation : [code/epsilon_contrainte.py](code/epsilon_contrainte.py).

Exemple : le front obtenu sur N0.

In [ ]:
import epsilon_contrainte

S0_N0 = json.load(open("donnees/S0_communs.json"))["N0"]
resultats, front = epsilon_contrainte.epsilon_contrainte(S0_N0, N0, temps_par_epsilon=0.5, graine=1)
print("Front (D, R) :", [(p[0], p[1]) for p in front])

**Front exact de N0.** N0 n'a que 8 interventions : on peut énumérer toutes les solutions. Pour chaque technicien et chaque sous-ensemble d'interventions, on cherche le meilleur ordre de passage réalisable, puis on essaie toutes les affectations (4⁸ cas). On obtient le vrai front de Pareto, qui sert à contrôler nos méthodes.

Implémentation : [code/exact_N0.py](code/exact_N0.py).

### 5.2 Recherche locale de Pareto (PLS)

La PLS garde une archive des solutions non dominées. Elle prend une solution de l'archive, explore tout son voisinage (relocate, échange de deux interventions et inversion d'un morceau de tournée) et ajoute à l'archive chaque voisin non dominé, en retirant les points qu'il domine. Elle s'arrête quand toute l'archive a été explorée ou au bout du temps imparti. Les distances sont comptées en mètres entiers pour éviter les erreurs d'arrondi. Le fichier contient aussi l'hypervolume et la mesure de couverture entre deux fronts.

Implémentation : [code/multiobjectif.py](code/multiobjectif.py).

Exemple : la PLS sur N0.

In [ ]:
import multiobjectif

res = multiobjectif.pls(S0_N0, N0, budget=5)
print("Front (D, R) :", [(p["D"], p["R"]) for p in res["front"]])

## 6. Campagnes d'expériences et analyses

Ces programmes réalisent les expériences présentées dans le rapport (cinq exécutions de 180 secondes par instance) puis refont les tableaux et les figures. Ils se lancent depuis un terminal, dans le dossier du notebook (par exemple `python code/campagne_recuit.py eps 0.1`).

**Recuit et ε-contrainte.** `campagne_recuit.py` lance l'analyse de sensibilité (un paramètre à la fois, puis des combinaisons), la campagne principale et la campagne ε-contrainte, et enregistre les résultats en JSON. `analyse_recuit.py` en tire les tableaux (meilleur coût, moyenne, écart-type, hypervolume) et les figures (convergence, sensibilité, fronts).

Implémentation : [code/campagne_recuit.py](code/campagne_recuit.py).

Implémentation : [code/analyse_recuit.py](code/analyse_recuit.py).

**Génétique et PLS.** `lancer.py` lance le génétique ou la PLS sur les instances choisies (`python code/lancer.py genetique --instances N50 --secondes 180`). `sensibilite_diversite.py` réalise le réglage du génétique (un paramètre à la fois, combinaisons, et test avec ou sans les mécanismes de diversité). `analyser.py` refait les tableaux et figures du génétique et de la comparaison PLS / ε-contrainte.

Implémentation : [code/lancer.py](code/lancer.py).

Implémentation : [code/sensibilite_diversite.py](code/sensibilite_diversite.py).

Implémentation : [code/analyser.py](code/analyser.py).

**Contrôle indépendant.** `verification_csv.py` recalcule le coût d'une solution en relisant directement les fichiers CSV, avec un code écrit séparément du socle ; une contrainte non respectée arrête le programme. `verifier.py` l'applique aux résultats du génétique et de la PLS, puis lance des essais très courts des deux méthodes.

Implémentation : [code/verification_csv.py](code/verification_csv.py).

Implémentation : [code/verifier.py](code/verifier.py).

## 7. Compléments ajoutés après la soutenance

### 7.1 Recuit simulé : P essais indépendants

Version du recuit qui suit le schéma du cours : P essais indépendants, chacun depuis une nouvelle solution aléatoire (obtenue par n mouvements relocate tirés au hasard depuis S₀) avec T = T₀. Un essai s'arrête après Q itérations sans amélioration ou à la fin de sa part de temps, et on garde la meilleure solution des P essais. Le budget total reste de 180 secondes. `comparer_P.py` compare cette version à notre recuit sur N50, N100 et N200 avec P = 1, 3 et 5.

Implémentation : [code/recuit_P.py](code/recuit_P.py).

Implémentation : [code/comparer_P.py](code/comparer_P.py).

### 7.2 Algorithme génétique : temps long et renouvellement

`experiences.py` regroupe les essais longs sur N200 (400 s), le recuit témoin (180 s) et la comparaison des trois politiques de renouvellement sur N50 (20 s).

La variante `genetique_renouvellement.py` permet de faire varier les deux taux d’immigration. Les paramètres des autres mécanismes sont conservés. `analyser_apports.py` recrée les tableaux et figures à partir des résultats sauvegardés. Voir le [guide d’utilisation](docs/UTILISATION.md) pour les commandes.

Implémentation : [code/genetique_renouvellement.py](code/genetique_renouvellement.py).

Implémentation : [code/experiences.py](code/experiences.py).

Implémentation : [code/analyser_apports.py](code/analyser_apports.py).
